<a href="https://colab.research.google.com/github/klavarihards/Markets-Strategies/blob/main/portfolio-engine/notebooks/04b_holdout_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The One-Time Hold-Out Evaluation

The last two years of data (23 Oct 2024 – 22 Oct 2026) were locked before any development started. Every method, setting, overlay and the final choice of portfolio per profile (`configs/final_choice.yaml`) were fixed on the earlier data, by rules written down in advance.

This notebook opens the hold-out **once**. Every walk-forward backtest is re-run on all the data with exactly the frozen settings, and only the hold-out dates are summarised. The opening is logged in `experiments/holdout_log.jsonl` (committed). **Nothing below may be used to change a method or a setting**; if anything is changed after this, these two years are no longer out of sample.

*Two years is short: a Sharpe ratio measured over two years is uncertain by roughly ±0.7, so the hold-out can reveal a broken method, but it cannot crown a winner.*

In [ ]:
# Setup: clone the repo (in Colab) and install the package, plus the libraries used for cross-checks.
import os, sys

IN_COLAB = "google.colab" in sys.modules
BRANCH = "claude/eloquent-fermat-i94z76"     # change to "main" once the branch is merged

if IN_COLAB:
    os.chdir("/content")
    if not os.path.exists("Markets-Strategies"):
        !git clone -q -b {BRANCH} https://github.com/klavarihards/Markets-Strategies.git
    else:
        !git -C Markets-Strategies fetch -q origin {BRANCH} && git -C Markets-Strategies checkout -q {BRANCH} && git -C Markets-Strategies pull -q origin {BRANCH}
    os.chdir("Markets-Strategies/portfolio-engine")
    !pip install -q -e ".[download,crosscheck]"
    sys.path.insert(0, "src")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working directory:", os.getcwd())

# In Colab, keep data snapshots and results on Google Drive: a Colab runtime's own files
# disappear when it restarts, and the snapshot must stay the same for every later run.
# The repo's data/snapshots and results folders become links to Drive, so nothing else changes.
USE_DRIVE = IN_COLAB
if USE_DRIVE:
    import shutil
    from google.colab import drive
    drive.mount("/content/drive")
    store = "/content/drive/MyDrive/portfolio-engine"
    for local, remote in [("data/snapshots", f"{store}/snapshots"), ("results", f"{store}/results")]:
        os.makedirs(remote, exist_ok=True)
        if not os.path.islink(local):
            if os.path.isdir(local):
                shutil.copytree(local, remote, dirs_exist_ok=True)   # keep anything already there
                shutil.rmtree(local)
            os.makedirs(os.path.dirname(local) or ".", exist_ok=True)
            os.symlink(remote, local)
    print("Snapshots and results are stored in", store)

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

from portfolio_engine.config import load_config, config_hash
from portfolio_engine.data import resolve_snapshot
from portfolio_engine.backtest.studies import evaluate_frozen_holdout

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 200)
cfg = load_config("configs/phase4_overlays.yaml")
choice = yaml.safe_load(open("configs/final_choice.yaml"))
SNAPSHOT = str(resolve_snapshot(cfg["data"]["snapshot_dir"]))
assert choice["based_on"]["config_hash"] == config_hash(cfg), "final_choice.yaml was made with different settings"
print("Frozen choice:", json.dumps(choice["portfolios"]))

In [ ]:
out = evaluate_frozen_holdout(cfg, choice, snapshot_dir=SNAPSHOT)
data, t = out["data"], out["tables"]
print(f"Hold-out: {data.holdout_start.date()} to {data.returns.index[-1].date()} | saved to {out['run_dir']}")
print(open("experiments/holdout_log.jsonl").read())

## 1. The frozen portfolios against the benchmarks, hold-out only

In [ ]:
cols = ["start", "end", "ann_return", "ann_vol", "sharpe", "max_drawdown", "cvar95_monthly", "turnover_per_year"]
fmt = {c: "{:.2%}" for c in cols if c not in ("start", "end", "sharpe")} | {"sharpe": "{:.2f}"}
t["frozen_vs_benchmarks"][cols].style.format(fmt)

In [ ]:
cmp_ = pd.concat({"development": t["frozen_dev_period"][["ann_return", "ann_vol", "sharpe", "max_drawdown"]],
                  "hold-out": t["frozen_vs_benchmarks"][["ann_return", "ann_vol", "sharpe", "max_drawdown"]]}, axis=1)
cmp_.style.format("{:.2f}")

In [ ]:
t["frozen_sharpe_ci"].style.format({"sharpe_diff": "{:+.2f}", "ci_low": "{:+.2f}", "ci_high": "{:+.2f}", "share_boot_positive": "{:.0%}"})

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
for (w, name), res in out["overlays"].items():
    if name in out["frozen"] or ":" not in name:
        r = res.returns.loc[data.holdout_start:]
        ax.plot((1 + r).cumprod(), label=name, linestyle="-" if ":" in name else "--")
ax.set_title("Hold-out period: frozen portfolios vs benchmarks (TWD, after costs)"); ax.grid(True, alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

## 2. Did the client limits hold?

In [ ]:
c = t["overlay_compliance"].reset_index()
c = c[c["method"].apply(lambda m: any(f"{p}:{m}" in out["frozen"] for p in ["conservative", "balanced", "growth"]))]
c.set_index(["profile", "method"])[["realised_vol", "vol_band_low", "vol_band_high", "vol_in_band", "max_drawdown",
                                    "max_loss_limit", "max_loss_breached"]].style.format(
    {"realised_vol": "{:.1%}", "vol_band_low": "{:.0%}", "vol_band_high": "{:.0%}", "max_drawdown": "{:.1%}", "max_loss_limit": "{:.0%}"})

## 3. Every method, hold-out only (core portfolios, both windows)

In [ ]:
t["core_summary"][["ann_return", "ann_vol", "sharpe", "max_drawdown", "turnover_per_year"]].style.format(
    {"ann_return": "{:.1%}", "ann_vol": "{:.1%}", "sharpe": "{:.2f}", "max_drawdown": "{:.1%}", "turnover_per_year": "{:.0%}"})

## 4. Did the overlays help out of sample? (Rule 1 re-checked on the hold-out)

In [ ]:
t["overlay_decisions_holdout"][["d_sharpe", "maxdd_improvement", "cvar_improvement", "risk_improvement", "keep"]].style.format(
    {"d_sharpe": "{:+.3f}", "maxdd_improvement": "{:+.0%}", "cvar_improvement": "{:+.0%}", "risk_improvement": "{:+.0%}"})

## Findings

*Written after this one-time run.*

- [RESULT]